In [ ]:
import medal
import pickle
import torch
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [4]:
os.getcwd()
sys.path.append('./lrp')

In [5]:
%reload_ext autoreload
%autoreload 2
import lrp 
import lrp_layers
import importlib
importlib.reload(lrp)
importlib.reload(lrp_layers)

<module 'lrp_layers' from '/insomnia001/depts/iicd/users/vj2308/phil1/./lrp/lrp_layers.py'>

In [ ]:
### note that lrp has been modified so that when you pass in model.model
### it will only access the encoder portion
# e.g. lrp_model=lrp.LRPModel(model.model)

In [5]:
### MNIST
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from tqdm import tqdm

X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X_scaled=X/255.0

train=X_scaled[:10000]
labels_train=y[:10000]
test=X_scaled[10000:10500]
labels_test=y[10000:10500]

In [36]:
def heatmap_features(fa_values_avg, title):
    '''
    This function plots a heatmap of the global feature attribution values
    Input: 
        fa_values: a list of 10 lists of averaged feature attribution values for each digit class
        title: str; title for the figure
    Output:
        Displays a 5 by 2 heatmap plot of global feature attribution values.
    '''
    
    fig, ax=plt.subplots(2,5,figsize=(10,5))
    for i in range(10):
        fa_val=fa_values_avg[i]/np.sum(fa_values_avg[i])
        
        ax[i//5, i%5].imshow(fa_val.reshape(28,28),cmap="turbo")
        ax[i//5, i%5].axis("off")
    fig.suptitle(title)

In [7]:
with open(f"MEDAL-o/model_mnist_tsne.pkl",'rb') as file:
    model=pickle.load(file)

In [ ]:
values_list=[np.zeros((1,784)) for i in range(10)]
lrp_model=lrp.LRPModel(model.model)

for i in range(10):
    for j in tqdm(train[labels_train==str(i)]):
        r=lrp_model.forward(torch.tensor(j).float().to('cuda'))
        values_list[i]+=r.detach().cpu().numpy()
for i in range(10):
    values_list[i]/=len(train[labels_train==str(i)])
    print(len(train[labels_train==str(i)]))

In [ ]:
heatmap_features(values_list,"LRP digits")